In [1]:
# Cell 1: Environment Setup

include("scripts/buildaux_helpers.jl")
include("scripts/buildaux_dictionaries.jl")
using .BuildAuxHelpers
using .BuildAuxDictionaries
using OMJulia

# --- Configuration ---

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the model to build
MODEL = "DIGrid"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"


"/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

In [2]:
# Cell 2: OpenModelica Setup + Single Model Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(omc, "loadModel(Complex)")
om_send(omc, "loadModel(ModelicaServices)")
om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the selected model and validate it
om_send(omc, "loadFile(\"$MODEL_FILE_PATH\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($MODEL)", parsed=false)
println(chk)


[ Info: Path to zmq file="/tmp/openmodelica.dyvulgawocfc.port.julia.HWnUGFlmh4"


OMC -> loadFile("/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/dyvulgawocfc/Notebooks ejemplo/BuildAux/models/DIGrid.mo")
OMC -> clearMessages()
OMC -> checkModel(DIGrid)
"Check of DIGrid completed successfully.
Class DIGrid has 242 equation(s) and 239 variable(s).
134 of these are trivial equation(s)."



In [3]:
# Cell 3: Auxiliary Model Setup

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE = joinpath(MODEL_DIR, AUX_MODEL * ".mo")


"/home/dyvulgawocfc/Notebooks ejemplo/BuildAux/models/DIGrid_auxiliary.mo"

In [4]:
# Cell 4: INIT / Optional Slack Configuration for the Single Model

INIT_MODEL_BY_COMPONENT = Dict{String, String}(
 #"generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = "inertialGrid1"


"inertialGrid1"

In [5]:
# Cell 5: Single-Model Auxiliary Build Pipeline

# Create/refresh the auxiliary model in OpenModelica
om_send(omc, "deleteClass($AUX_MODEL)")
om_send(omc, "clearMessages()")
om_send(omc, "copyClass($MODEL, \"$AUX_MODEL\")")

# Build component dictionary from the source model
components = get_all_components(omc, MODEL)

# Apply dictionary-driven replacements
apply_replacements!(omc, MODEL, AUX_MODEL, REPLACEMENTS, components, SLACK_COMPONENT)

# Delete connections to cleanup targets
delete_connections!(omc, AUX_MODEL, components)

# Delete cleanup-target components
delete_components!(omc, AUX_MODEL, components)

# Add INIT models for the source model
add_init_models!(omc, MODEL, AUX_MODEL, INIT_MODELS, INIT_MODEL_BY_COMPONENT, components, SLACK_COMPONENT)

# Add load-flow modifiers
apply_LF_modifiers!(omc, MODEL, AUX_MODEL, INIT_MODELS, components)

# Add initial equations for the source model
add_init_equations!(omc, MODEL, AUX_MODEL, components, INIT_MODELS, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)

# Save the auxiliary model
om_send(omc, "saveModel(\"$AUX_FILE\", $AUX_MODEL)")
patch_aux_equations!(AUX_FILE, SLACK_COMPONENT; components = components)

# Re-load the patched auxiliary model and validate the build
om_send(omc, "deleteClass($AUX_MODEL)")
om_send(omc, "loadFile(\"$AUX_FILE\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($AUX_MODEL)", parsed=false)
println(chk)


OMC -> deleteClass(DIGrid_auxiliary)
OMC -> clearMessages()
OMC -> copyClass(DIGrid, "DIGrid_auxiliary")
OMC -> updateComponent(inertialGrid2, Dynawo.Electrical.Machines.Simplified.GeneratorAlphaBeta, DIGrid_auxiliary, modification = $Code((PGen0Pu = 3.3, Alpha = 0, Beta = 0, QGen0Pu = 0, U0Pu = 1, u0Pu = Complex(1, 0), i0Pu = Modelica.ComplexMath.conj(Complex(inertialGrid2.PGen0Pu, inertialGrid2.QGen0Pu) / inertialGrid2.u0Pu))))
OMC -> updateComponent(inertialGrid1, Dynawo.Electrical.Buses.InfiniteBus, DIGrid_auxiliary, modification = $Code((UPu = 1, UPhase = 0)))
OMC -> addComponent(load_INIT, Dynawo.Electrical.Loads.Load_INIT, DIGrid_auxiliary, modification = $Code((Q0Pu = 0, P0Pu = 5, U0Pu(start = 1, fixed = false), UPhase0(start = 0, fixed = false))))
OMC -> addComponent(loadPQ_INIT, Dynawo.Electrical.Loads.Load_INIT, DIGrid_auxiliary, modification = $Code((Q0Pu = 0, P0Pu = 0, U0Pu(start = 1, fixed = false), UPhase0(start = 0, fixed = false))))
OMC -> updateComponent(load, Dynawo.

### Optional diagnostics for the single-file build
Run the next cell only if the main build/check cell fails or you need detailed OpenModelica messages.


In [6]:
# Cell 7: OMC diagnostics for failed checks

# Run this cell after the build/check cell to isolate OpenModelica failures.

function _print_omc_errors(label::String)
    raw = String(sendExpression(omc, "getErrorString()", parsed=false))
    txt = strip(replace(raw, "\"" => ""))
    println("\n[$label] getErrorString()")
    if isempty(txt)
        println("<no messages>")
    else
        println(raw)
    end
end

function _check_and_report(model_name::String)
    sendExpression(omc, "clearMessages()")
    println("\n=== checkModel($model_name) ===")
    chk = sendExpression(omc, "checkModel($model_name)", parsed=false)
    println(chk)
    _print_omc_errors(model_name)
    return chk
end

println("=== OMC diagnostics start ===")
_print_omc_errors("after previous cell")

# 1) Auxiliary model
_check_and_report(AUX_MODEL)

# 2) Optional compile-time expansion (often gives clearer errors)
sendExpression(omc, "clearMessages()")
println("\n=== instantiateModel($AUX_MODEL) ===")
inst = sendExpression(omc, "instantiateModel($AUX_MODEL)", parsed=false)
inst_s = String(inst)
if startswith(strip(inst_s), "Error")
    println(inst_s)
end
_print_omc_errors("instantiateModel")

println("=== OMC diagnostics end ===")


=== OMC diagnostics start ===

[after previous cell] getErrorString()
"Notification: Modelica requested package Complex of version 3.2.3. Complex 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
Notification: Modelica requested package ModelicaServices of version 3.2.3. ModelicaServices 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
[/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:20:3-20:98:writable] Warning: Connector switchOffSignal1 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:21:3-21:125:writable] Warning: Connector switchOffSignal2 is not balanced: The number of potential variables (1) is not equal to the numb

In [7]:
# Cell 8: Trace load INIT mode selection

# Run this cell to inspect how BuildAux classifies each Dynawo load in MODEL.

components = get_all_components(omc, MODEL)
found_load = false

println("=== Load INIT mode trace for $MODEL ===")

for (comp_name, c) in sort(collect(components), by = first)
    base_class = c["class"]::String
    startswith(base_class, "Dynawo.Electrical.Loads.") || continue

    found_load = true
    mode = BuildAuxBuild._load_init_mode(omc, MODEL, components, comp_name, base_class)

    raw_mods = c["modifiers"]
    has_direct_complex =
        isa(raw_mods, Dict) &&
        haskey(raw_mods, "s0Pu") &&
        haskey(raw_mods, "u0Pu") &&
        haskey(raw_mods, "i0Pu")

    p_ref = strip(resolve_load_ref_value(omc, MODEL, comp_name, "PRefPu"))
    q_ref = strip(resolve_load_ref_value(omc, MODEL, comp_name, "QRefPu"))

    println("\n$comp_name")
    println("  class = $base_class")
    println("  mode = $mode")
    println("  has s0Pu/u0Pu/i0Pu = $has_direct_complex")
    println("  PRefPu source = ", isempty(p_ref) ? "<none>" : p_ref)
    println("  QRefPu source = ", isempty(q_ref) ? "<none>" : q_ref)
end

if !found_load
    println("No Dynawo load components found in $MODEL")
end

println("\n=== End load INIT mode trace ===")


=== Load INIT mode trace for DIGrid ===

load
  class = Dynawo.Electrical.Loads.LoadZIP
  mode = pq_init
  has s0Pu/u0Pu/i0Pu = false
  PRefPu source = 5
  QRefPu source = 0

loadPQ
  class = Dynawo.Electrical.Loads.LoadPQ
  mode = pq_init
  has s0Pu/u0Pu/i0Pu = false
  PRefPu source = 0
  QRefPu source = 0

=== End load INIT mode trace ===
